# Neo4j Knowledge Graph


In [ ]:
%pip install -q \
  torch==2.13.0 \
  transformers==5.16.1 \
  datasets==5.0.1 \
  accelerate==1.14.0 \
  peft==0.20.0 \
  trl==1.12.0 \
  bitsandbytes==0.50.2 \
  evaluate==0.4.6 \
  requests tqdm sentencepiece huggingface_hub pandas scikit-learn

%pip install -q neo4j

In [ ]:
import json, random, re, string, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from datasets import Dataset, DatasetDict

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_REPO="https://github.com/Gokcimen/Home_Appliance_Dataset"
!rm -rf /content/Home_Appliance_Dataset
!git clone -q --depth 1 {DATA_REPO}.git /content/Home_Appliance_Dataset

DATA_ROOT=Path("/content/Home_Appliance_Dataset")

def flatten(path):
    raw=json.loads(Path(path).read_text(encoding="utf-8"))
    rows=[]
    for article in raw["data"]:
        title=article["title"]
        for para in article["paragraphs"]:
            context=para["context"]
            for qa in para["qas"]:
                rows.append({
                    "id":str(qa["id"]),
                    "title":title,
                    "context":context,
                    "question":qa["question"],
                    "answers":{
                        "text":[a["text"] for a in qa["answers"]],
                        "answer_start":[int(a["answer_start"]) for a in qa["answers"]],
                    }
                })
    return rows

raw_datasets=DatasetDict({
    "train":Dataset.from_list(flatten(DATA_ROOT/"train.json")),
    "validation":Dataset.from_list(flatten(DATA_ROOT/"dev.json")),
    "test":Dataset.from_list(flatten(DATA_ROOT/"test.json")),
})

assert len(raw_datasets["train"])==8000
assert len(raw_datasets["validation"])==1000
assert len(raw_datasets["test"])==1000

ids={s:set(raw_datasets[s]["id"]) for s in raw_datasets}
assert not ids["train"]&ids["validation"]
assert not ids["train"]&ids["test"]
assert not ids["validation"]&ids["test"]
all_ids=set().union(*ids.values())
assert len(all_ids)==10000
assert {int(x) for x in all_ids}==set(range(1,10001))

titles={s:set(raw_datasets[s]["title"]) for s in raw_datasets}
assert not titles["train"]&titles["validation"]
assert not titles["train"]&titles["test"]
assert not titles["validation"]&titles["test"]
assert len(set().union(*titles.values()))==1111

print("train",len(raw_datasets["train"]))
print("validation",len(raw_datasets["validation"]))
print("test",len(raw_datasets["test"]))
print("products",len(set().union(*titles.values())))


In [ ]:
products={}
questions={}
answers={}
pq_edges=[]
qa_edges=[]
pid_by_title={}
pid_no=0
q_no=0

for split in ["train","validation","test"]:
    for row in raw_datasets[split]:
        title=row["title"]
        if title not in pid_by_title:
            pid_no+=1
            pid_by_title[title]=f"P{pid_no:04d}"
            products[pid_by_title[title]]={"title":title,"split":split}
        pid=pid_by_title[title]

        q_no+=1
        qid=f"Q{q_no:05d}"
        aid=f"A{q_no:05d}"

        questions[qid]={
            "dataset_id":row["id"],
            "question":row["question"],
            "product_id":pid,
            "split":split,
        }
        answers[aid]={
            "text":row["answers"]["text"][0],
            "question_id":qid,
            "split":split,
        }
        pq_edges.append((pid,qid))
        qa_edges.append((qid,aid))

stats={
    "Primary product/entity nodes":len(products),
    "Question nodes":len(questions),
    "Answer nodes":len(answers),
    "Total nodes":len(products)+len(questions)+len(answers),
    "Product/entity-question edges":len(pq_edges),
    "Question-answer edges":len(qa_edges),
    "Total edges":len(pq_edges)+len(qa_edges),
}
print(stats)

assert stats=={
    "Primary product/entity nodes":1111,
    "Question nodes":10000,
    "Answer nodes":10000,
    "Total nodes":21111,
    "Product/entity-question edges":10000,
    "Question-answer edges":10000,
    "Total edges":20000,
}


In [ ]:
def structured_facts(title,context):
    facts=[f"Product: {title}"]
    sentences=[x.strip() for x in re.split(r"(?<=[.!?])\s+",context) if x.strip()]
    keywords=("capacity","energy","efficiency","noise","dimension","weight","feature",
              "technology","warranty","program","kg","litre","liter","db")
    for s in sentences:
        if any(k in s.lower() for k in keywords):
            facts.append(s)
    return "\n".join(facts)

print(structured_facts(raw_datasets["test"][0]["title"],raw_datasets["test"][0]["context"]))


In [ ]:
import os
from neo4j import GraphDatabase

if os.getenv("NEO4J_URI"):
    driver=GraphDatabase.driver(
        os.environ["NEO4J_URI"],
        auth=(os.environ["NEO4J_USER"],os.environ["NEO4J_PASSWORD"])
    )
    with driver.session() as session:
        session.run("CREATE CONSTRAINT product_id IF NOT EXISTS FOR (p:Product) REQUIRE p.id IS UNIQUE")
        session.run("CREATE CONSTRAINT question_id IF NOT EXISTS FOR (q:Question) REQUIRE q.id IS UNIQUE")
        session.run("CREATE CONSTRAINT answer_id IF NOT EXISTS FOR (a:Answer) REQUIRE a.id IS UNIQUE")
    print("Neo4j connection ready")
